# RAV-Med — Step 2b: Does the 100-token encoder limit handicap similarity on HealthContradict?

**Why.** In Step 2 the SBERT encoder used its default `max_seq_length = 100` tokens, so **72.9%** of the 80-word windows were truncated *for the similarity term only* (the NLI verifier saw them in full). If truncation hurts similarity, the Step 2 comparisons *similarity-only vs ERS* are biased against similarity. This notebook tests that directly.

**Variants (fixed before running; the scorer, reviewed claims, documents and irrelevant-document sample are identical to Step 2; bootstrap draws use a fresh seed, so intervals can differ in the third decimal):**
| variant | window | SBERT limit | what it isolates |
|---|---|---|---|
| `B_80w_enc256` | 80 words (as Step 2) | **256 tokens** | only the truncation changes |
| `A_50w_enc100` | **50 words** | 100 tokens (as Step 2) | windows that fit the original limit |

The Step 2 result (`80w_enc100`) is the reference; its numbers are hard-coded below for the comparison table.

**Pre-specified reading.** The Step 2 conclusions are *robust* if, with the reference claim, (i) ERS still beats similarity-only (paired question-level test), (ii) ERS is still ≈ contradiction-only, (iii) the irrelevant-document AUROC of similarity stays above ERS, and (iv) the label-free claim still inverts on *no* questions. Any change is reported, whichever direction it goes.

**Needs:** `hc_claims.csv` from the Step 2 zip (the reviewed claims; upload when asked). No LLM is loaded, so a free T4 session is enough (minutes, since the contradiction model runs only on each document's top-3 windows; results are cached and resume after a disconnect).

## 1. Install

In [ ]:
!pip install -q sentence-transformers transformers accelerate scikit-learn scipy

## 2. Configuration

In [ ]:
import os, re, json, random, hashlib, subprocess
import numpy as np, pandas as pd, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda"
EMBED_MODEL = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
NLI_MODEL   = "roberta-large-mnli"
VARIANTS = [("B_80w_enc256", 80, 256), ("A_50w_enc100", 50, 100)]   # (name, window words, SBERT max_seq_length)
MAX_WORDS_PER_DOC = 9600     # same document coverage as Step 2 (120 windows x 80 words)
MIN_CHUNK_WORDS = 15
TOP_K = 3
N_IRRELEVANT_PER_TOPIC = 5
N_BOOT = 2000
BATCH_SIZE = 64
FULL_SENSITIVITY = False   # False: run the contradiction model only on the top-3 windows (by similarity) of each document. The primary
                           # endpoint (top-1 window), contradiction-only and the top-3 mean are exactly the same as scoring every window;
                           # only the "max-S minus max-C" sensitivity needs all windows, so it is skipped (~40x less compute).

# resume after a Colab disconnect: variant results are cached; try Google Drive, fall back to local disk
CACHE = "/content/hc2_cache"
try:
    from google.colab import drive; drive.mount("/content/drive"); CACHE = "/content/drive/MyDrive/hc2_cache"
except Exception as e: print("Drive not available, using local cache:", str(e)[:80])
os.makedirs(CACHE, exist_ok=True); print("cache dir:", CACHE)

# Step 2 reference results (80-word windows, SBERT limit 100): pooled pairwise accuracy, from hc_table_pairwise_accuracy.csv
STEP2 = {("ref","Similarity only"):0.6913, ("ref","Contradiction only"):0.8011, ("ref","Full ERS"):0.8043,
         ("free","Similarity only"):0.4554, ("free","Contradiction only"):0.4261, ("free","Full ERS"):0.4272}
STEP2_AUROC = {("ref","Similarity only"):0.9772, ("ref","Contradiction only"):0.7743, ("ref","Full ERS"):0.8672,
               ("free","Similarity only"):0.9775, ("free","Contradiction only"):0.6505, ("free","Full ERS"):0.7850}

## 3. Load HealthContradict and the reviewed claims

In [ ]:
if not os.path.exists("HealthContradict"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tinaboya/HealthContradict"], check=True)
rows = [json.loads(l) for l in open("HealthContradict/dataset/dataset_ready.jsonl", encoding="utf-8")]
raw = pd.DataFrame(rows); raw["topic_id"] = raw["topic_id"].astype(int)
raw["correct_text"]   = np.where(raw.query_stance == "yes", raw.doc_a, raw.doc_b)
raw["incorrect_text"] = np.where(raw.query_stance == "yes", raw.doc_b, raw.doc_a)
def did(t): return hashlib.md5(t.encode("utf-8")).hexdigest()[:12]
raw["correct_id"] = raw.correct_text.map(did); raw["incorrect_id"] = raw.incorrect_text.map(did)
docs = {}
for col in ["correct", "incorrect"]:
    for i, t in zip(raw[f"{col}_id"], raw[f"{col}_text"]): docs[i] = t
pairs = raw[["instance_id", "topic_id", "query", "query_stance", "correct_id", "incorrect_id"]].copy()
topics = pairs.drop_duplicates("topic_id").set_index("topic_id")[["query", "query_stance"]]
L = lambda ids: pd.Series([len(docs[i].split()) for i in ids])
print(f"{len(pairs)} pairs | {len(topics)} questions | {len(docs)} unique documents")

if not os.path.exists("hc_claims.csv"):
    from google.colab import files
    print("Upload hc_claims.csv from the Step 2 zip"); files.upload()
claims = pd.read_csv("hc_claims.csv").set_index("topic_id")
assert len(claims) == 81 and set(claims.index) == set(topics.index), "claims file does not match the 81 questions"
print("claims loaded:", len(claims))

## 4. Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print("SBERT default max_seq_length:", embedder.max_seq_length)
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE).eval()
c_idx = [i for i, l in nli.config.id2label.items() if l.lower().startswith("contra")][0]

@torch.no_grad()
def contra(prem, hyp, bs=BATCH_SIZE):
    order = np.argsort([len(p) for p in prem]); out = np.zeros(len(prem))
    for i in range(0, len(prem), bs):
        idx = order[i:i+bs]
        enc = nli_tok([prem[j] for j in idx], [hyp[j] for j in idx], truncation=True, padding=True, max_length=256, return_tensors="pt").to(DEVICE)
        out[idx] = torch.softmax(nli(**enc).logits, -1)[:, c_idx].cpu().numpy()
    return out

## 5. Scoring function (identical to Step 2 apart from window size and encoder limit)

In [ ]:
def build_chunks(chunk_words, need):
    max_chunks = MAX_WORDS_PER_DOC // chunk_words
    def chunk(text):
        w = text.split()
        ch = [" ".join(w[i:i+chunk_words]) for i in range(0, len(w), chunk_words)]
        if len(ch) > 1 and len(ch[-1].split()) < MIN_CHUNK_WORDS: ch = ch[:-1]
        return ch[:max_chunks] if ch else [text]
    rows, trunc = [], 0
    doc_ids = sorted(need.doc_id.unique())
    for d in doc_ids:
        if len(docs[d].split()) > MAX_WORDS_PER_DOC: trunc += 1
        for k, c in enumerate(chunk(docs[d])): rows.append((d, k, c))
    return pd.DataFrame(rows, columns=["doc_id", "k", "text"]), trunc, len(doc_ids)

def make_need_and_irr():
    # same draw order as Step 2 so the irrelevant documents are identical
    rng = np.random.default_rng(SEED)
    all_by_topic = {}
    for _, r in pairs.iterrows(): all_by_topic.setdefault(r.topic_id, set()).update([r.correct_id, r.incorrect_id])
    irr_rows = []
    for t in topics.index:
        other = sorted(set().union(*[v for k, v in all_by_topic.items() if k != t]) - all_by_topic[t])
        for d in rng.choice(other, size=N_IRRELEVANT_PER_TOPIC, replace=False): irr_rows.append((t, d))
    irr = pd.DataFrame(irr_rows, columns=["topic_id", "doc_id"])
    need = pd.concat([pairs[["topic_id"]].assign(doc_id=pairs.correct_id), pairs[["topic_id"]].assign(doc_id=pairs.incorrect_id), irr]).drop_duplicates()
    return need, irr, rng
need, irr, rng = make_need_and_irr()

def score_variant(name, chunk_words, enc_len):
    embedder.max_seq_length = enc_len
    chunks, trunc, n_docs = build_chunks(chunk_words, need)
    sample = chunks.text.sample(min(2000, len(chunks)), random_state=SEED).tolist()
    n_sb  = np.array([len(embedder.tokenizer.tokenize(t)) + 2 for t in sample])     # +2 for [CLS]/[SEP]
    n_nli = np.array([len(nli_tok.tokenize(t)) for t in sample])
    info = dict(variant=name, window_words=chunk_words, sbert_limit=enc_len, n_docs=n_docs, n_windows=len(chunks), docs_truncated=trunc,
                frac_windows_over_sbert_limit=float((n_sb > enc_len).mean()), frac_windows_over_nli_limit=float((n_nli > 250).mean()),
                median_sbert_tokens=float(np.median(n_sb)))
    print(info)
    emb = embedder.encode(chunks.text.tolist(), batch_size=BATCH_SIZE, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
    cidx = chunks.reset_index().groupby("doc_id")["index"].apply(list).to_dict()
    cemb = {(t, ct): embedder.encode([claims.loc[t, f"claim_{ct}"]], normalize_embeddings=True, convert_to_numpy=True)[0]
            for t in claims.index for ct in ("ref", "free")}
    out = []
    for ct in ("ref", "free"):
        meta, prem, hyp = [], [], []
        for t, d in need.itertuples(index=False):
            ids = cidx[d]; S = emb[ids] @ cemb[(t, ct)]
            keep = np.arange(len(ids)) if FULL_SENSITIVITY else np.argsort(-S, kind="stable")[:TOP_K]
            for q in keep: j = ids[q]; meta.append((t, d, S[q])); prem.append(chunks.text[j]); hyp.append(claims.loc[t, f"claim_{ct}"])
        m = pd.DataFrame(meta, columns=["topic_id", "doc_id", "S"]); m["C"] = contra(prem, hyp)
        for (t, d), g in m.groupby(["topic_id", "doc_id"], sort=False):
            g = g.sort_values("S", ascending=False); top = g.iloc[0]; topk = g.iloc[:TOP_K]
            out.append(dict(topic_id=t, doc_id=d, claim_type=ct, S_top1=top.S, C_top1=top.C, ERS_top1=top.S - top.C,
                            ERS_topk=(topk.S - topk.C).mean(), ERS_maxC=(g.S.max() - g.C.max()) if FULL_SENSITIVITY else np.nan, n_windows=len(cidx[d])))
    sc = pd.DataFrame(out); sc.to_csv(f"{CACHE}/hc2_doc_scores_{name}.csv", index=False)
    json.dump(info, open(f"{CACHE}/hc2_info_{name}.json", "w"))
    return sc, info

## 6. Analysis function (same endpoints and cluster bootstrap as Step 2)

In [ ]:
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score
tid = pairs.topic_id.values; utid = np.unique(tid); tpos = {t: np.where(tid == t)[0] for t in utid}
st = pairs.query_stance.values
def win(a, b): return (a > b).astype(float) + 0.5 * (a == b)
_rng = np.random.default_rng(SEED); _ = _rng.choice(np.arange(5), size=1)  # placeholder to keep stream separate
boot_t = np.random.default_rng(SEED + 1).integers(0, len(utid), size=(N_BOOT, len(utid)))

def per_q(w):  return np.array([w[tpos[t]].mean() for t in utid])
def boot_pooled(w):
    s = np.array([w[tpos[t]].sum() for t in utid]); n = np.array([len(tpos[t]) for t in utid]); b = s[boot_t].sum(1) / n[boot_t].sum(1)
    return w.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)
def boot_qw(w):
    p = per_q(w); b = p[boot_t].mean(1); return p.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)

def analyze(scores, tag):
    S_ = scores.set_index(["topic_id", "doc_id", "claim_type"])
    def ps(ct, scorer):
        c = S_.loc[list(zip(pairs.topic_id, pairs.correct_id, [ct] * len(pairs)))]; i = S_.loc[list(zip(pairs.topic_id, pairs.incorrect_id, [ct] * len(pairs)))]
        if scorer == "similarity": return c.S_top1.values, i.S_top1.values
        if scorer == "contradiction": return -c.C_top1.values, -i.C_top1.values
        return c[scorer].values, i[scorer].values
    names = {"similarity": "Similarity only", "contradiction": "Contradiction only", "ERS_top1": "Full ERS", "ERS_topk": "ERS top-3 mean", "ERS_maxC": "ERS max-S minus max-C"}
    main, W = [], {}
    for ct in ("ref", "free"):
        for sc, nm in names.items():
            if sc == "ERS_maxC" and scores["ERS_maxC"].isna().all(): continue
            w = win(*ps(ct, sc)); W[(ct, sc)] = w
            p, lo, hi = boot_pooled(w); q, qlo, qhi = boot_qw(w)
            main.append(dict(variant=tag, claim=ct, scorer=nm, pooled=p, lo=lo, hi=hi, q_weighted=q, qw_lo=qlo, qw_hi=qhi))
    main = pd.DataFrame(main)
    # paired differences (pooled, cluster bootstrap) and question-level Wilcoxon
    diffs = []
    for ct in ("ref", "free"):
        for a, b, lab in [("ERS_top1", "similarity", "ERS - similarity"), ("ERS_top1", "contradiction", "ERS - contradiction")]:
            d = W[(ct, a)] - W[(ct, b)]; m, lo, hi = boot_pooled(d)
            pq = per_q(W[(ct, a)]) - per_q(W[(ct, b)]); p = wilcoxon(pq, alternative="two-sided").pvalue if np.any(pq != 0) else 1.0
            qm, qlo, qhi = boot_qw(d)
            diffs.append(dict(variant=tag, claim=ct, contrast=lab, pooled_diff=m, lo=lo, hi=hi, qw_diff=qm, qw_lo=qlo, qw_hi=qhi, wilcoxon_p_question_level=p))
        pq = per_q(W[(ct, "ERS_top1")]) - 0.5
        diffs.append(dict(variant=tag, claim=ct, contrast="ERS vs chance 0.5", pooled_diff=W[(ct, "ERS_top1")].mean() - 0.5, lo=np.nan, hi=np.nan,
                          qw_diff=pq.mean(), qw_lo=np.nan, qw_hi=np.nan, wilcoxon_p_question_level=wilcoxon(pq, alternative="two-sided").pvalue))
    diffs = pd.DataFrame(diffs)
    # by gold answer
    byans = []
    for ct in ("ref", "free"):
        for stance in ("yes", "no"):
            m = st == stance
            for sc, nm in [("similarity", "Similarity only"), ("ERS_top1", "Full ERS")]:
                w = W[(ct, sc)]; tt = np.unique(tid[m]); pos = [np.where((tid == t) & m)[0] for t in tt]
                sums = np.array([w[p].sum() for p in pos]); ns = np.array([len(p) for p in pos]); bi = np.random.default_rng(SEED + 2).integers(0, len(tt), size=(N_BOOT, len(tt)))
                b = sums[bi].sum(1) / ns[bi].sum(1)
                byans.append(dict(variant=tag, claim=ct, gold_answer=stance, scorer=nm, n_pairs=int(m.sum()), n_questions=len(tt), pairwise_acc=w[m].mean(), lo=np.percentile(b, 2.5), hi=np.percentile(b, 97.5)))
    byans = pd.DataFrame(byans)
    # irrelevant-document control
    cor = pairs[["topic_id", "correct_id"]].drop_duplicates().rename(columns={"correct_id": "doc_id"}); airr = []
    for ct in ("ref", "free"):
        for col, nm in [("S_top1", "Similarity only"), ("C_top1", "Contradiction only"), ("ERS_top1", "Full ERS")]:
            g1 = cor.merge(scores[scores.claim_type == ct], on=["topic_id", "doc_id"]); g0 = irr.merge(scores[scores.claim_type == ct], on=["topic_id", "doc_id"])
            sign = -1 if col == "C_top1" else 1
            y = np.r_[np.ones(len(g1)), np.zeros(len(g0))]; x = np.r_[sign * g1[col], sign * g0[col]]; tq = np.r_[g1.topic_id, g0.topic_id]
            r2 = np.random.default_rng(SEED + 3); bs = []
            for _ in range(1000):
                pick = r2.choice(utid, size=len(utid), replace=True); idx = np.concatenate([np.where(tq == t)[0] for t in pick])
                if len(np.unique(y[idx])) == 2: bs.append(roc_auc_score(y[idx], x[idx]))
            airr.append(dict(variant=tag, claim=ct, scorer=nm, AUROC=roc_auc_score(y, x), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5)))
    airr = pd.DataFrame(airr)
    # threshold transfer
    thr = []
    for ct in ("ref", "free"):
        c, i = ps(ct, "ERS_top1")
        for T in (0.0, 0.2, 0.4):
            thr.append(dict(variant=tag, claim=ct, T=T, correct_doc_SAFE=(c >= T).mean(), incorrect_doc_SAFE=(i >= T).mean()))
    return main, diffs, byans, airr, pd.DataFrame(thr)

## 7. Run the variants (each variant is saved as soon as it finishes)

In [ ]:
results, infos = {}, []
for name, cw, el in VARIANTS:
    f = f"{CACHE}/hc2_doc_scores_{name}.csv"
    if os.path.exists(f):
        sc = pd.read_csv(f); info = json.load(open(f"{CACHE}/hc2_info_{name}.json")); print(f"{name}: loaded from cache (resume)")
    else:
        sc, info = score_variant(name, cw, el)
    infos.append(info)
    results[name] = analyze(sc, name)
    print(f"--- {name} done"); print(results[name][0].round(3).to_string(index=False))
pd.DataFrame(infos).to_csv("hc2_variant_info.csv", index=False)
print(pd.DataFrame(infos).round(3).to_string(index=False))

## 8. Comparison with Step 2 and the pre-specified reading

In [ ]:
cat = lambda k: pd.concat([results[n][k] for n in results], ignore_index=True)
MAIN, DIFF, BYANS, AIR, THR = [cat(k) for k in range(5)]
for nm, d in [("main", MAIN), ("diffs", DIFF), ("by_answer", BYANS), ("irrelevant", AIR), ("threshold", THR)]: d.to_csv(f"hc2_table_{nm}.csv", index=False)

print("PAIRWISE ACCURACY (pooled), Step 2 vs variants")
rows = []
for (ct, nm), v in STEP2.items():
    r = dict(claim=ct, scorer=nm, step2_80w_enc100=v)
    for name in results: r[name] = MAIN[(MAIN.variant == name) & (MAIN.claim == ct) & (MAIN.scorer == nm)].pooled.values[0]
    rows.append(r)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nPAIRED CONTRASTS (reference claim) and question-level Wilcoxon p"); print(DIFF[DIFF.claim == "ref"].round(3).to_string(index=False))
print("\nBY GOLD ANSWER"); print(BYANS.round(3).to_string(index=False))
print("\nIRRELEVANT-DOCUMENT AUROC"); print(AIR.round(3).to_string(index=False))
print("\nTHRESHOLD TRANSFER"); print(THR.round(3).to_string(index=False))

## 9. Save

In [ ]:
import shutil
os.makedirs("rav_med_step2b_outputs", exist_ok=True)
cands = [f for f in os.listdir(".") if f.startswith("hc2_")] + [os.path.join(CACHE, f) for f in os.listdir(CACHE)]
for f in cands:
    if os.path.isfile(f): shutil.copy(f, "rav_med_step2b_outputs")      # skip the hc2_cache folder itself
shutil.make_archive("rav_med_step2b_outputs", "zip", "rav_med_step2b_outputs")
try:
    from google.colab import files; files.download("rav_med_step2b_outputs.zip")
except ImportError: pass
print("Saved rav_med_step2b_outputs.zip")